# SI — two-colour DNA-PAINT with spectrally close dyes: ATTO 655 + CF640R (Ximea)

**Makes up:** the SI figure of the BSC-1 cell imaged with ATTO 655 and CF640R imager strands
(`Massive_Cells_Render_Box_CF640RATTO655`, with its FRC inset).

**Data:** BioStudies `S-BIAD3210`, `2026_Multicolour_Paper/Ximea/Revision_2/20260923 Massive Cell Red/`
`F1_250pM_CF640R_F2_750pM_ATTO655_Diluted/30mW_638_1_DC1_NF_785sp_1/`: raw Ximea TIFFs (30 files,
~120 GB) plus the acquisition's MicroManager metadata (the diluted acquisition of this prep, the
published cell). The deposition holds raw data only, so this notebook fits it first.

The two dyes' emission peaks are ~17 nm apart: A_R 0.66 (ATTO 655) vs 0.73 (CF640R) expected,
against a single blink's A_R error of ~0.03. Events are first clustered in position and colour
(`unmix_channels_joint_cluster`, as for Cy3B / ATTO 594), but that method's own two-component fit
never separates this pair (most clusters hold 3–9 events). So the dyes' mean colours come from a
two-component fit to the clusters with ≥ 20 events, and every cluster and isolated event is
classified against them (`_dna_paint_cells.classify_from_large_clusters`).

## FAST_MODE

`FAST_MODE = True` (default) downloads and fits only the first TIFF (~4 GB), enough to run the
pipeline end to end with far fewer localisations than the published figure (the classification
needs enough large clusters, so it may be poor). `FAST_MODE = False` downloads and fits the whole
acquisition (~120 GB; many hours; run in `tmux`/`screen`).

---
## 0 — Imports

In [ ]:
import json
import sys
import types
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# _biostudies_download.py and _dna_paint_cells.py live one level up (notebooks/figures/).
sys.path.insert(0, str(Path.cwd().resolve().parent))
import _biostudies_download as bd
import _dna_paint_cells as dpc

from pyS3M import IOFunctions
from pyS3M import sCMOSFunctions
from pyS3M import SpectralFunctions
from pyS3M import MaskFunctions
from pyS3M import SpotDetectionFunctions
from pyS3M import SR_Functions
from pyS3M import PlottingBase
from pyS3M import FiducialDetection
from pyS3M import SM_extractionfunctions
from pyS3M.DriftCorrectionFunctions import Drift_Correction_Functions

IO = IOFunctions.IO_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
plotter = PlottingBase.PublicationPlotter(dark_background=False)
FD = FiducialDetection.FiducialDetector()
SM_E = SM_extractionfunctions.extract_SMs()
drift_corrector = Drift_Correction_Functions()

print("Python:", sys.version)

---
## 1 — Configuration

In [ ]:
FAST_MODE = True   # True: first TIFF only (~4 GB, illustrative). False: all 30 TIFFs (~120 GB, the published data).
REFIT = False      # True: fit again even if a fit with the same files and settings exists.

CAMERA = "ximea"
PIXEL_SIZE_NM = 69.0   # sample-plane pixel (src/CameraDefaults.py)

REPO_ROOT = Path.cwd().resolve().parents[2]   # notebooks/figures/SI -> repo root
CALIB_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT  = Path("outputs") / "FigureSI_DNAPAINT_CF640R_ATTO655"   # git-ignored (bare outputs/ rule)
FIG_DIR   = OUT_ROOT / ("fast" if FAST_MODE else "full")
# Point DATA_DIR at a drive with room for the download. One folder per acquisition and per
# FAST_MODE setting: the fit reads every TIFF inside it.
DATA_DIR  = OUT_ROOT / ("data_fast" if FAST_MODE else "data_full")
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Every camera-dependent class set to this camera (SuperRes_Functions' spot-detection
# sub-instance does not take the camera from its own argument, so it is passed in).
S_F  = SpectralFunctions.Spectral_Funcs(camera=CAMERA)
M_F  = MaskFunctions.Mask_Functions(camera=CAMERA)
SD_F = SpotDetectionFunctions.SpotDetection_Functions(camera=CAMERA)
SupRes_F = SR_Functions.SuperRes_Functions(camera=CAMERA, mask_functions=M_F, spot_detection_functions=SD_F)

gain_map   = IO.read_tiff(CALIB_DIR / "gain.tif")
offset_map = IO.read_tiff(CALIB_DIR / "offset.tif")
variance   = IO.read_tiff(CALIB_DIR / "variance.tif")
read_noise = IO.read_tiff(CALIB_DIR / "readnoise.tif")
rqe        = IO.read_tiff(CALIB_DIR / "rqe.tif")

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

print(f"FAST_MODE  : {FAST_MODE}")
print(f"Data dir   : {DATA_DIR.resolve()}")
print(f"Figures dir: {FIG_DIR.resolve()}")

---
## 2 — Download the raw data from BioStudies (`S-BIAD3210`)

Re-running is safe: complete files are skipped and interrupted ones resume.

In [ ]:
REMOTE_FOLDER = ("2026_Multicolour_Paper/Ximea/Revision_2/20260923 Massive Cell Red/"
                 "F1_250pM_CF640R_F2_750pM_ATTO655_Diluted/30mW_638_1_DC1_NF_785sp_1")

downloaded = bd.download_acquisition(REMOTE_FOLDER, DATA_DIR, first_file_only=FAST_MODE)

# Camera ROI of the acquisition (the fitted xc/yc are relative to its corner).
roi_x, roi_y, width, height = IO.metadata_reader_imageJ(downloaded["metadata"][0])
print(f"ROI: x={roi_x} y={roi_y}, {width} x {height} px; {len(downloaded['images'])} TIFF(s)")

---
## 3 — Spot detection and fitting

The paper's settings: pfa 1e-3, PSF peak wavelength 0.68 µm, 14 px ROI. A record of what was fitted is saved
beside `Localisations.h5`, so re-running reuses the fit unless the files or settings change (or `REFIT = True`).

In [ ]:
FIT_PARAMS = dict(pfa=1e-3, peak_wavelength=0.68, ROI_size=14, NA=1.49, pixel_size=PIXEL_SIZE_NM / 1000)

# Quick single-frame check of the spot detection before the full fit.
fig, axs = SupRes_F.example_spots_singleframe(
    DATA_DIR, pfa=FIT_PARAMS["pfa"], frame_index=10, variance=variance, read_noise=read_noise, rqe=rqe,
    offset_map=offset_map, gain_map=gain_map, peak_wavelength=FIT_PARAMS["peak_wavelength"],
    ROI_size=FIT_PARAMS["ROI_size"])
plt.show()

fit_file = DATA_DIR / "Localisations.h5"
fit_record_file = DATA_DIR / "Localisations_fit_record.json"
fit_record = {"images": [p.name for p in downloaded["images"]], **FIT_PARAMS}
previous = json.loads(fit_record_file.read_text()) if fit_record_file.exists() else None
if REFIT or not fit_file.exists() or previous != fit_record:
    SupRes_F.fit_imaging_data(DATA_DIR, smoothing_function, gain_map, offset_map, rqe, read_noise,
                              variance=variance, image_type=".tif", **FIT_PARAMS)
    fit_record_file.write_text(json.dumps(fit_record, indent=1))
else:
    print(f"Reusing {fit_file} (same files and settings)")

---
## 4 — Quality cuts, drift correction, linking, fiducials

As for Cy3B / ATTO 594 (`_dna_paint_cells.quality_cuts`): localisation and PSF-width uncertainty < 40 nm, PSF width
75–250 nm, χ²<sub>ν</sub> < 3, colour errors < 1; AIM drift correction, linking, fiducial removal.

In [ ]:
loc_data = pd.read_hdf(fit_file)
kept = dpc.quality_cuts(loc_data, PIXEL_SIZE_NM)
print(f"{len(loc_data):,} localisations, {len(kept):,} after the quality cuts ({len(kept) / len(loc_data):.0%})")

linked, linked_fremoved, drift = dpc.drift_link_fiducials(kept, width, height, PIXEL_SIZE_NM, drift_corrector, FD)
n_frames = int(linked_fremoved["frame"].max()) + 1
print(f"{len(linked):,} linked events, {len(linked_fremoved):,} after fiducial removal")

fig, ax = plotter.one_column_plot(width=3, height=1.6)
ax.plot(np.asarray(drift.drift_x) * PIXEL_SIZE_NM, label="x")
ax.plot(np.asarray(drift.drift_y) * PIXEL_SIZE_NM, label="y")
ax.set_xlabel("frame"); ax.set_ylabel("drift (nm)"); ax.legend(fontsize=7)
ax.grid(True, ls="--", color="0.85", lw=0.5)
plt.show()

---
## 5 — Separate ATTO 655 and CF640R

1. Joint spatial-spectral clustering with the Cy3B / ATTO 594 settings (`d_threshold = 1.0`, at least 3 events).
2. Its clusters (not its channels) are classified: a two-component GMM on the inverse-variance mean
   colours of clusters with ≥ 20 events gives the dyes' means; each cluster and isolated event is
   assigned at posterior ≥ 0.90, its own measurement variance added to each component.

In [ ]:
joint_locs, joint_meta = SM_E.unmix_channels_joint_cluster(
    linked_fremoved, n_channels=2, channels_to_use=["A_R", "A_G"], spatial_cols=["xc", "yc"],
    spatial_err_cols=["xc_err", "yc_err"], d_threshold=1.0, min_cluster_size=3,
    confidence_threshold_isolated=0.90, verbose=True, plot_results=True)
plt.show()

assigned, cls_info = dpc.classify_from_large_clusters(joint_locs, min_events=20, conf_threshold=0.90)
print(f"{joint_meta['n_clusters']:,} clusters, {cls_info['n_large']:,} with >= 20 events "
      f"({cls_info['large_events'] / len(assigned):.1%} of events)")
for k, dye in enumerate(("ATTO 655", "CF640R")):
    print(f"  {dye}: mean A_R {cls_info['means'][k, 0]:.4f}, A_G {cls_info['means'][k, 1]:.4f} "
          f"(weight {cls_info['weights'][k]:.2f})")

---
## 6 — Name the channels

The lower mean A_R is ATTO 655 (cyan), the higher CF640R (hot).

In [ ]:
DYES = ("ATTO 655", "CF640R")
channel_0, channel_1 = dpc.name_channels(assigned, DYES)

---
## 7 — Render

ATTO 655 in cyan, CF640R in hot, drawn at its printed size (~4.9 in).

In [ ]:
OVERSAMPLING = 8
img_cyan, img_hot = dpc.render_channels([channel_0, channel_1], width, height, n_frames, PIXEL_SIZE_NM,
                                        oversampling=OVERSAMPLING)
fig, ax = dpc.plot_render(plotter, img_cyan, img_hot, PIXEL_SIZE_NM, OVERSAMPLING, size_in=5, dyes=DYES)
plt.savefig(FIG_DIR / "Massive_Cells_Render_Box_CF640RATTO655.svg", dpi=600, format="svg")
plt.show()

---
## 8 — FRC (inset)

FIRE of each channel from 50 half-splits of its events (8x zoom); the band is the +-1 sigma spread.

In [ ]:
frc = [dpc.compute_fire(ch[["xc", "yc"]].to_numpy(), PIXEL_SIZE_NM) for ch in (channel_0, channel_1)]
for dye, r in zip(DYES, frc):
    print(f"{dye}: FIRE {r['resolution_nm']:.1f} nm (band {min(r['res_lo_nm'], r['res_hi_nm']):.1f}-"
          f"{max(r['res_lo_nm'], r['res_hi_nm']):.1f} nm)")
fig, ax = dpc.plot_frc(plotter, frc, DYES)
plt.savefig(FIG_DIR / "FRC_CF640RATTO655.svg", dpi=600, format="svg")
plt.show()